# battle_ship キャラ立ち絵ジェネレーター（Google Colab 無料 GPU）

艦カードの全身立ち絵を生成し、背景を切り抜いて `portraits.zip` で持ち帰ります。キャラクターデザインそのものに専念するため、艤装・エフェクト・背景は描かせず、描かれてしまった候補は自動で落とします。中身は `tools/portraitgen`（手元で `go run ./cmd/portraits kit` すると prompts と一緒に `kit.zip` にまとまります）。

**流れ**

1. メニュー **ランタイム → ランタイムのタイプを変更 → T4 GPU**
2. ① で `kit.zip` をアップロード
3. ③ **探索**: カードごとに下描き（832×1216）を何枚も描き、自動で選別
   - 不採用: 全身が枠からはみ出す・顔が小さすぎる／大きすぎる・複数人・エフェクトや背景（キラキラ、炎、月、空、艤装などをタガーで検出）・背景が無地でない・キャラから離れた物体がある
   - 合格した候補はスコア順（美観スコア＋デザイン通りか〔髪色・武器〕−黒つぶれの強いコントラスト）
4. ④ **仕上げ**: 採用した下描きを Real-ESRGAN で拡大してから 1.5 倍で描き直し（hires）、キャラ全体をタイルに分けて約 2 倍で描き直し（衣装・装飾・武器の細部）、顔と手を 1024px で描き直し、最後にもう一度エフェクト混入を検査して切り抜き
5. ⑤ で `portraits.zip` をダウンロード → 手元で `cd backend && go run ./cmd/portraits import ~/Downloads/portraits.zip`

T4 の目安: 下描き 1 枚 約 30 秒、仕上げ 1 枚 約 3 分。探索は選別が厳しい分だけ多めに描くので全艦で 1 時間強、仕上げが約 1 時間です。途中で切断されても続きから再開できるよう、`SAVE_TO_DRIVE` で Google ドライブに保存できます（おすすめ）。

In [ ]:
#@title ① セットアップ（kit.zip をアップロード）
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
#@markdown ドライブに前回の kit.zip があればそれを使います。prompts を変えたときはチェックして再アップロード
REUPLOAD_KIT = False  #@param {type:"boolean"}

import os, shutil, subprocess, sys
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/battle_ship_portraits"
else:
    WORK = "/content/portraits"
os.makedirs(WORK, exist_ok=True)

saved = f"{WORK}/kit.zip"
if REUPLOAD_KIT or not os.path.exists(saved):
    from google.colab import files
    up = files.upload()
    with open(saved, "wb") as f:
        f.write(next(iter(up.values())))
shutil.rmtree("/content/kit", ignore_errors=True)
shutil.unpack_archive(saved, "/content/kit", "zip")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "/content/kit"], check=True)

from portraitgen import run
JOBS = run.Job.load("/content/kit/jobs.json")
BY_ID = {j.id: j for j in JOBS}
print("作業フォルダ:", WORK)
print(f"{len(JOBS)} 枚:", ", ".join(BY_ID))

In [ ]:
#@title ② モデル読み込み（初回はダウンロードに数分）
#@markdown 既定は Illustrious 系の WAI（ソシャゲ風の塗りに強い）。diffusers 形式の SDXL なら差し替え可（例: `cagliostrolab/animagine-xl-4.0`）
MODEL_ID = "John6666/wai-nsfw-illustrious-sdxl-v150-sdxl"  #@param {type:"string"}
from portraitgen.sdxl import Generator
gen = Generator(MODEL_ID)
print("ready:", MODEL_ID)
print(gen.memory_report())  # 正常なら全部 float16、VRAM 約 7 GiB（拡大用の Real-ESRGAN も読み込み済み）

In [ ]:
#@title ③ 探索: 下描きを描いて自動選別
#@markdown `WANT` 枚合格するまで最大 `MAX_TRIES` 枚描きます（描き済みはスキップ）。気に入った候補がなければ `SEED_OFFSET` を変えて再実行すると別の絵が増えます
WANT = 4  #@param {type:"integer"}
MAX_TRIES = 16  #@param {type:"integer"}
SEED_OFFSET = 0  #@param {type:"integer"}
#@markdown 空欄なら全カード。例: `bb_kurogane, dd_asanagi`
ONLY = ""  #@param {type:"string"}

from IPython.display import display
only = [s.strip() for s in ONLY.split(",") if s.strip()]
for s in only:
    assert s in BY_ID, f"kit に {s} がありません"
targets = [BY_ID[s] for s in only] if only else JOBS
run.explore(gen, targets, WORK, want=WANT, max_tries=MAX_TRIES, seed_offset=SEED_OFFSET)

# 左から良い順。数字は seed とスコア、黄色は減点理由（タガーが武器を見つけられない・影が強い等）。赤いものは自動で不採用になった候補と理由
for job in targets:
    print(f"\n{job.id}  {job.name}   おすすめ: {run.best(WORK, job.id)}")
    sheet = run.candidate_sheet(WORK, job.id)
    if sheet:
        display(sheet)

In [ ]:
#@title ④ 仕上げ: 描き直し・顔と手の修正・切り抜き
#@markdown `PICKS` は `カードID=seed` をカンマ区切りで（例: `bb_guren=1234, ca_soyo=5678`）。指定しないカードはおすすめ（スコア最上位）を使います
PICKS = ""  #@param {type:"string"}
ONLY = ""  #@param {type:"string"}
#@markdown 仕上げ済みでも作り直す
REDO = False  #@param {type:"boolean"}
#@markdown 描き直しの強さ。上げるほど細部が描き込まれるが、下描きから離れる（TILE は衣装・武器の細部。0 で省略）
REFINE_STRENGTH = 0.35  #@param {type:"slider", min:0.2, max:0.6, step:0.05}
TILE_STRENGTH = 0.3  #@param {type:"slider", min:0.0, max:0.5, step:0.05}
FACE_STRENGTH = 0.4  #@param {type:"slider", min:0.2, max:0.6, step:0.05}
HAND_STRENGTH = 0.35  #@param {type:"slider", min:0.2, max:0.6, step:0.05}

picks = {}
for item in filter(None, (s.strip() for s in PICKS.split(","))):
    card_id, seed = item.split("=")
    assert card_id.strip() in BY_ID, f"kit に {card_id} がありません"
    picks[card_id.strip()] = int(seed)
only = [s.strip() for s in ONLY.split(",") if s.strip()]
targets = [BY_ID[s] for s in only] if only else JOBS
run.finish(gen, targets, WORK, picks=picks, redo=REDO,
           refine_strength=REFINE_STRENGTH, tile_strength=TILE_STRENGTH,
           face_strength=FACE_STRENGTH, hand_strength=HAND_STRENGTH)

# ゲームの背景色に並べてプレビュー（赤枠はカードやマップの駒の顔の位置合わせに使う顔の検出結果）
sheet = run.final_sheet(WORK)
if sheet:
    display(sheet)

In [ ]:
#@title ⑤ portraits.zip をダウンロード
from google.colab import files
files.download(str(run.pack(WORK, "/content/portraits.zip")))